# Day 34 — Functions: `def`, `return`, `*args`, Scope
**Month 3 | Week 1 Gap-Fill | Pure Python**

> **Real-world framing:**
> Every reusable analysis tool you write — cleaning pipelines, scoring functions,
> report generators — is a function. If you can't write a clean function, you can't
> build a pipeline. If you can't build a pipeline, every project starts from zero.
>
> Today you learn to package logic. By the end, you'll write a function that cleans
> a column, another that scores data quality, and a third that accepts any number of
> metrics and formats them into a business summary — the exact pattern used in
> production ETL and client dashboards.

---

**Skills used today:** Python basics (Day 32), Loops + Conditionals (Day 33)  
**New today:** `def`, `return`, default parameters, `*args`, `**kwargs`, scope (LEGB),
`lambda` (preview only — full lesson Day 54), docstrings, type hints (light intro)

---


---
## 📦 Section 1 — Raw Data (Read Only — Never Modify This Section)

ShopEase transaction snippet — same fictional dataset you've been working with.
All tasks in this notebook operate on copies of these objects, never on the originals.


In [2]:
# ── RAW DATA — DO NOT MODIFY BELOW THIS LINE ──────────────────────────────

raw_transactions = [
    {"order_id": "O001", "customer": "Amit",    "category": "Electronics", "revenue": 12500, "quantity": 2, "region": "North", "month": "Jan"},
    {"order_id": "O002", "customer": "Priya",   "category": "Clothing",    "revenue":  3200, "quantity": 4, "region": "South", "month": "Jan"},
    {"order_id": "O003", "customer": "Rahul",   "category": "Electronics", "revenue": 18900, "quantity": 1, "region": "West",  "month": "Feb"},
    {"order_id": "O004", "customer": "Sneha",   "category": "Groceries",   "revenue":  1100, "quantity": 8, "region": "North", "month": "Feb"},
    {"order_id": "O005", "customer": "Karan",   "category": "Clothing",    "revenue":  4700, "quantity": 3, "region": "East",  "month": "Mar"},
    {"order_id": "O006", "customer": "Divya",   "category": "Electronics", "revenue": 22000, "quantity": 1, "region": "South", "month": "Mar"},
    {"order_id": "O007", "customer": "Mohit",   "category": "Groceries",   "revenue":   850, "quantity": 6, "region": "West",  "month": "Apr"},
    {"order_id": "O008", "customer": "Anjali",  "category": "Clothing",    "revenue":  5500, "quantity": 2, "region": "North", "month": "Apr"},
    {"order_id": "O009", "customer": "Vikram",  "category": "Electronics", "revenue": 31000, "quantity": 2, "region": "East",  "month": "May"},
    {"order_id": "O010", "customer": "Neha",    "category": "Groceries",   "revenue":  1400, "quantity": 5, "region": "South", "month": "May"},
    {"order_id": "O011", "customer": "Arjun",   "category": "Clothing",    "revenue":  2900, "quantity": None, "region": "West",  "month": "Jun"},  # intentional None
    {"order_id": "O012", "customer": "meera",   "category": "electronics", "revenue": 15600, "quantity": 3, "region": "North", "month": "Jun"},  # lowercase issue
    {"order_id": "O013", "customer": "Suresh",  "category": "Groceries",   "revenue": -500,  "quantity": 2, "region": "East",  "month": "Jul"},  # negative revenue
    {"order_id": "O014", "customer": "Kavita",  "category": "Clothing",    "revenue":  6800, "quantity": 1, "region": "South", "month": "Jul"},
    {"order_id": "O015", "customer": "Rohit",   "category": "Electronics", "revenue": 9900,  "quantity": 4, "region": "West",  "month": "Aug"},
]

# Verify — print first 3 rows
for row in raw_transactions[:3]:
    print(row)


{'order_id': 'O001', 'customer': 'Amit', 'category': 'Electronics', 'revenue': 12500, 'quantity': 2, 'region': 'North', 'month': 'Jan'}
{'order_id': 'O002', 'customer': 'Priya', 'category': 'Clothing', 'revenue': 3200, 'quantity': 4, 'region': 'South', 'month': 'Jan'}
{'order_id': 'O003', 'customer': 'Rahul', 'category': 'Electronics', 'revenue': 18900, 'quantity': 1, 'region': 'West', 'month': 'Feb'}


---
## 📖 Section 2 — Concept Notes

Read this section completely before attempting any tasks.

---

### 2.1 — Why Functions Exist

Without functions, every time you need to clean a column you copy-paste 8 lines of code.
With a function, you call `clean_column(df['category'])` once and it works everywhere.

**Functions let you:**
- Reuse logic without copy-pasting
- Test one unit in isolation (unit testing)
- Give logic a name that explains intent (`calculate_ltv()` is infinitely clearer than 50 lines of math)
- Build pipelines: `output = step3(step2(step1(raw_data)))`

---

### 2.2 — Anatomy of a Function

```python
def function_name(param1, param2="default"):
    """Docstring: what this function does, what it returns."""
    # logic here
    result = param1 + param2
    return result
```

| Part | Purpose |
|------|---------|
| `def` | Declares a function |
| `function_name` | Snake_case always. Verb + noun: `clean_revenue`, `flag_nulls` |
| `param1` | Required parameter — caller MUST provide it |
| `param2="default"` | Optional parameter — has a fallback value |
| `return` | Sends a value back to the caller. Without `return`, function returns `None` |
| Docstring | First string inside the function, triple-quoted. Always write one. |

---

### 2.3 — Return vs Print

```python
# THIS IS WRONG FOR PRODUCTION CODE
def get_total(nums):
    print(sum(nums))          # prints but returns Nothing

# THIS IS CORRECT
def get_total(nums):
    return sum(nums)          # caller can use the value

total = get_total([100, 200, 300])   # total = 600
doubled = total * 2                  # works because we returned
```

**Rule:** `print()` is for humans. `return` is for code. Production functions always `return`.

---

### 2.4 — `*args` — Accept Any Number of Positional Arguments

```python
def summarise(*metrics):
    """Accept any number of metric names and print them."""
    for m in metrics:
        print(f"Metric: {m}")

summarise("revenue", "quantity")           # 2 args
summarise("revenue", "quantity", "profit") # 3 args — same function
```

`*args` collects all positional arguments into a **tuple**. Use when the caller decides
how many values to pass.

---

### 2.5 — `**kwargs` — Accept Any Number of Keyword Arguments

```python
def build_report(**sections):
    """Build a report from named sections."""
    for name, content in sections.items():
        print(f"[{name.upper()}]")
        print(content)

build_report(
    summary="Revenue up 12%",
    recommendation="Invest in North region"
)
```

`**kwargs` collects keyword arguments into a **dict**. Less common than `*args` but
essential when building config-driven tools.

---

### 2.6 — Scope: LEGB Rule

Python looks up variable names in this exact order:

| Letter | Scope | Example |
|--------|-------|---------|
| **L** | Local | Variable defined inside the current function |
| **E** | Enclosing | Variable in the outer function (for nested functions) |
| **G** | Global | Variable defined at module level |
| **B** | Built-in | Python built-ins: `len`, `range`, `print` |

```python
revenue = 100_000   # Global

def calculate_tax(rate=0.18):
    tax = revenue * rate    # 'tax' is Local; 'revenue' is Global (found at G)
    return tax

print(tax)   # NameError — 'tax' only exists inside the function
```

**Rule:** Never rely on global variables inside functions. Pass everything as parameters.
This makes functions testable and reusable.

---

### 2.7 — Common Mistakes

| Mistake | What happens | Fix |
|---------|-------------|-----|
| `print()` instead of `return` | Caller gets `None` | Always `return` |
| Forgetting `return` | Function silently returns `None` | Add `return result` |
| Modifying input list in-place | Original data changes | Work on a copy: `data = data.copy()` |
| Global variable inside function | Function breaks when moved to another file | Pass as parameter |
| Mutable default argument: `def f(lst=[])` | Bug: list persists across calls | Use `def f(lst=None): if lst is None: lst = []` |

---

### 2.8 — Interview Frame

> *"How would you explain functions to a client asking why their pipeline broke?"*
>
> "A function is a reusable contract. It takes defined inputs, does one job, and returns
> a defined output. If your pipeline broke, it means one function received unexpected input
> or returned None where a value was expected. Functions make bugs findable and fixable — 
> without them, you'd be debugging 500 lines of flat code."

---


---
## ✏️ Section 3 — Practice Tasks

**Rules:**
1. Work on copies of `raw_transactions` — never modify the original
2. Write plain-English comments before writing code (comment-skeleton habit)
3. Every function must have a docstring
4. Every function must `return`, not `print`
5. Test every function with at least one call after defining it

---

### Task A — Basic Functions (25 pts)


#### A1 — Revenue Tier Classifier (8 pts)

Write a function `classify_revenue(amount, low_threshold=5000, high_threshold=15000)`.

Rules:
- Returns `"Low"` if amount < low_threshold
- Returns `"High"` if amount >= high_threshold
- Returns `"Mid"` otherwise
- If amount is `None` or negative, return `"Invalid"`

Test it with: `1100`, `4700`, `12500`, `22000`, `None`, `-500`


In [2]:
# A1 — classify_revenue
def classify_revenue(amount, low_threshold=5000, high_threshold=15000):
    """
    Classify a revenue amount into Low / Mid / High tier.
    
    Parameters
    ----------
    amount : int or float or None
        Revenue value to classify.
    low_threshold : int, default 5000
        Values below this are 'Low'.
    high_threshold : int, default 15000
        Values at or above this are 'High'.
    
    Returns
    -------
    str : 'Low', 'Mid', 'High', or 'Invalid'
    """
    # Guard: handle None and negative values first
    if amount is None or amount < 0:
        return "Invalid"
    # Check thresholds in order
    if amount < low_threshold:
        return "Low"
    if amount >= high_threshold:
        return "High"
    return "Mid"

# Test
test_values = [1100, 4700, 12500, 22000, None, -500]
for v in test_values:
    print(f"classify_revenue({v}) → {classify_revenue(v)}")




classify_revenue(1100) → Low
classify_revenue(4700) → Low
classify_revenue(12500) → Mid
classify_revenue(22000) → High
classify_revenue(None) → Invalid
classify_revenue(-500) → Invalid


#### A2 — Average Revenue Calculator (7 pts)

Write a function `avg_revenue(transactions, category=None)`.

- `transactions` = the list of dicts (raw_transactions)
- If `category` is None: return average revenue across ALL transactions
- If `category` is provided (e.g. `"Electronics"`): return average for that category only
- Skip rows where revenue is None or negative
- Return the result **rounded to 2 decimal places**

Test it with: no category (all), `"Electronics"`, `"Groceries"`, `"clothing"` (lowercase — should still work, case-insensitive)


In [4]:
# A2 — avg_revenue
def avg_revenue(transactions, category=None):
    """" Calculate average revenue across all or filtered transactions.

    Parameters
    ----------
    transactions : list of dict
    category : str or None
        If provided, filter to this category (case=insensitive)

    Returns
    -------
    float : average revenue rounded to 2 decimal places
    """
    # Filter by category if provided (case-insensitive)
    if category is not None:
        rows = [r for r in transactions if r['category'].lower() == category.lower()]
    else:
        rows = transactions

    # Collect valid revenue values (skip None and negative)
    valid = [r['revenue'] for r in rows if r['revenue'] is not None and r['revenue'] >= 0]

    # Guard: empty list
    if not valid:
        return 0.0

    return round(sum(valid) / len(valid), 2)

# Test
print(f"All categories:  ₹{avg_revenue(raw_transactions):,.2f}")
print(f"Electronics:  ₹{avg_revenue(raw_transactions, 'Electronics'):,.2f}") 
print(f"Clothing:     ₹{avg_revenue(raw_transactions, 'Clothing'):,.2f}")
print(f"Groceries:    ₹{avg_revenue(raw_transactions, 'Groceries'):,.2f}")
print(f"clothing (lowercase): ₹{avg_revenue(raw_transactions, 'clothing'):,.2f}")

All categories:  ₹9,739.29
Electronics:  ₹18,316.67
Clothing:     ₹4,620.00
Groceries:    ₹1,116.67
clothing (lowercase): ₹4,620.00


#### A3 — Row Formatter (10 pts)

Write a function `format_row(order_id, customer, revenue, region, currency="₹", width=40)`.

It should return a **formatted string** (not print) like this:

```
O001 | Amit          | ₹12,500 | North
```

Rules:
- Customer name padded to 12 characters (left-aligned)
- Revenue formatted with comma separator and currency prefix
- Width parameter is for the full output string — pad with spaces if shorter
- Function must `return` the string

Test with: O001/Amit/12500/North, O009/Vikram/31000/East


In [5]:
# A3 — format_row
def format_row(order_id, customer, revenue, region, currency='₹', width=40):
    """ 
    Format a single transaction row as a display string.
    
    Paramenters
    ----------
    order_id : str
    customer : str
    revenue : int or float
    region : str
    currency : str, default '₹'
    width    : int, default 40 - minimum total width (padded with spaces)
    
    Returns
    -------
    str : formated row string
    """
    # Build each part
    name_padded = customer.ljust(12)
    revenue_str = f"{currency}{revenue:,.0f}"
    
    # Assemble the row
    row = f"{order_id} | {name_padded} | {revenue_str} | {region}"

    # Pad to minimum width
    return row.ljust(width)

# Test
print(repr(format_row("O001", "Amit", 12500, "North")))
print(repr(format_row("O002", "Priya", 3200, "East")))
print(format_row("O001", "Amit", 12500, "North"))
print(format_row("O009", "Vikram", 31000, "East"))

'O001 | Amit         | ₹12,500 | North   '
'O002 | Priya        | ₹3,200 | East     '
O001 | Amit         | ₹12,500 | North   
O009 | Vikram       | ₹31,000 | East    


---
### Task B — *args and **kwargs (25 pts)


#### B1 — Multi-Metric Aggregator with *args (12 pts)

Write a function `aggregate_metrics(transactions, *fields)`.

- `transactions` = list of dicts
- `*fields` = any number of field names to aggregate (e.g. `"revenue"`, `"quantity"`)
- For each field: compute sum, mean, min, max — skip None and negative values
- Return a **dict** structured as:

```python
{
  "revenue": {"sum": ..., "mean": ..., "min": ..., "max": ...},
  "quantity": {"sum": ..., "mean": ..., "min": ..., "max": ...}
}
```

Test with: `aggregate_metrics(raw_transactions, "revenue")` and 
`aggregate_metrics(raw_transactions, "revenue", "quantity")`


In [5]:
# B1 — aggregate_metrics

def aggregate_metrics(transactions, *fields):
    """
    Aggregate any number of numeric fields from transaction list.
    
    Parameters
    ----------
    transactions : list of dict
    *fields : str
       Any number of field names to aggregate.
       
    returns
    -------
    dict : {field: {sum, mean, min, max}} for each field
    """
    result = {}
    for field in fields:
        # Collect valid values ( skip None and negative for numeric fields)
        values = [
            row[field]
            for row in transactions
            if row.get(field) is not None and row[field] >=0
        ]

        if not values:
            result[field] = {"sum": 0 , "mean": 0, "min": 0, "max": 0}
            continue
        result[field] = {
            "sum": sum(values),
            "mean": round(sum(values) / len(values), 2),
            "min": min(values),
            "max": max(values)
        }

    return result

# Test 1: Single field
r1 = aggregate_metrics(raw_transactions, "revenue")
print("Single field = revenue:")
for k, v in r1.items():
        print(f" {k}: {v}")

# Test 2 "two fields
r2 = aggregate_metrics(raw_transactions, "revenue", "quantity")
print("\nTwo fields:")
for k, v in r2.items():
    print(f" {k}: {v}")

Single field = revenue:
 revenue: {'sum': 136350, 'mean': 9739.29, 'min': 850, 'max': 31000}

Two fields:
 revenue: {'sum': 136350, 'mean': 9739.29, 'min': 850, 'max': 31000}
 quantity: {'sum': 44, 'mean': 3.14, 'min': 1, 'max': 8}


#### B2 — Report Builder with **kwargs (13 pts)

Write a function `build_summary_report(title, **sections)`.

- `title` = report title string
- `**sections` = any number of named sections (e.g. `summary="..."`, `finding="..."`, `action="..."`)
- Returns a formatted multi-line string (not print) that looks like:

```
════════════════════════════════════════
 SHOPEASE MONTHLY ANALYSIS
════════════════════════════════════════
[SUMMARY]
Revenue up 18% vs last month — Electronics led growth.

[FINDING]
North region contributes 42% of total revenue.

[ACTION]
Allocate 20% more budget to North Electronics in Q3.
════════════════════════════════════════
```

Test with at least 3 sections. Print the returned string using `print()`.


In [6]:
# B2 — build_summary_report

def build_summary_report(title , **sections):
    """
    Build a formated text report from a title and named sections.
    
    Parameters
    ----------
    title : str
        Report title.
    **sections : str
        Any number of named content sections.
        
    Returns
    -------
    str : formated report string
    """
    
    separator = "=" * 44
    lines =[]
    lines.append(separator)
    lines.append(f" {title.upper()}")
    lines.append(separator)
    
    for section_name, content in sections.items():
        lines.append(f"\n[{section_name.upper()}]")
        lines.append(content)

    lines.append(separator)
    
    return "\n".join(lines)

# Test - 3sections
report = build_summary_report(
    "ShopEase Monthly Analysis",
    summary = 'Revenue up 18% vs last month - Electronics led growth.',
    finding = 'North region contributes 42% of total revenue.',
    action = 'Allocate 20% more budget to North Electronics in Q3.'
)
print(report)


 SHOPEASE MONTHLY ANALYSIS

[SUMMARY]
Revenue up 18% vs last month - Electronics led growth.

[FINDING]
North region contributes 42% of total revenue.

[ACTION]
Allocate 20% more budget to North Electronics in Q3.


---
### Task C — Scope + Data Quality Function (20 pts)


#### C1 — Data Quality Scorer (20 pts)

Write a function `score_data_quality(transactions)` that:

1. Counts total rows
2. Counts rows where `revenue` is None or negative → flag as `revenue_issues`
3. Counts rows where `quantity` is None → flag as `quantity_nulls`
4. Counts rows where `customer` name is lowercase (not title case) → flag as `name_issues`
5. Counts rows where `category` is not one of: `Electronics`, `Clothing`, `Groceries` (case-sensitive) → flag as `category_issues`
6. Computes an overall quality score: `((total - total_issues) / total) * 100` rounded to 1 decimal
7. Returns a **dict** with all counts + the quality score

Then write a second function `print_quality_report(quality_dict)` that:
- Accepts the dict returned by `score_data_quality`
- Prints a formatted quality report
- Flags each issue count in plain English with NRA format:
  - Number: how many issues
  - Reason: what kind of issue
  - Action: what to fix

Call both functions on `raw_transactions`. There are **4 intentional issues** — find all 4.


In [18]:
# C1 — score_data_quality + print_quality_report
VALID_CATEGORIES = {"Electronics", "Clothing", "Groceries"}

def score_data_quality(transactions):
    """
    Assess data quality across 4 dimensions.
    
    Returns
    -------
    dict with counts and overall quality score (0-100)
    """
    total = len(transactions)
    revenue_issues  = 0
    quantity_nulls  = 0
    name_issues     = 0
    category_issues = 0
    
    for row in transactions:
        # Revenue: None or negative
        if row["revenue"] is None or row["revenue"] < 0:
            revenue_issues += 1
        
        # Quantity: None
        if row["quantity"] is None:
            quantity_nulls += 1
        
        # Customer name: not title case (first letter not uppercase)
        if row["customer"] != row["customer"].title():
            name_issues += 1
        
        # Category: not in approved set (case-sensitive)
        if row["category"] not in VALID_CATEGORIES:
            category_issues += 1
    
    total_issues = revenue_issues + quantity_nulls + name_issues + category_issues
    quality_score = round(((total - total_issues) / total) * 100, 1)
    
    return {
        "total_rows":       total,
        "revenue_issues":   revenue_issues,
        "quantity_nulls":   quantity_nulls,
        "name_issues":      name_issues,
        "category_issues":  category_issues,
        "total_issues":     total_issues,
        "quality_score":    quality_score,
    }


def print_quality_report(quality_dict):
    """
    Print a formatted NRA-style quality report from the quality dict.
    """
    q = quality_dict
    print("=" * 50)
    print(" DATA QUALITY REPORT — ShopEase Transactions")
    print("=" * 50)
    print(f"Total rows scanned: {q['total_rows']}")
    print(f"Overall quality score: {q['quality_score']}%")
    print()
    
    issues = [
        ("Revenue Issues",  q["revenue_issues"],  "revenue",  "Negative or null revenue",   "Flag and quarantine — investigate source system"),
        ("Quantity Nulls",  q["quantity_nulls"],   "quantity", "Missing quantity values",    "Backfill from order logs or default to 0"),
        ("Name Issues",     q["name_issues"],      "customer", "Non-title-case customer name","Standardise at ingestion: .title() on all names"),
        ("Category Issues", q["category_issues"],  "category", "Unlisted or lowercase category","Add validation at point of entry"),
    ]
    
    for label, count, field, reason, action in issues:
        if count > 0:
            status = "⚠️ FLAGGED"
        else:
            status = "✅ CLEAN"
        print(f"{status} {label}: {count} row(s)")
        if count > 0:
            print(f"   → Reason: {reason}")
            print(f"   → Action: {action}")
    
    print("=" * 50)


# Run on raw data
quality = score_data_quality(raw_transactions)
print_quality_report(quality)




 DATA QUALITY REPORT — ShopEase Transactions
Total rows scanned: 15
Overall quality score: 73.3%

⚠️ FLAGGED Revenue Issues: 1 row(s)
   → Reason: Negative or null revenue
   → Action: Flag and quarantine — investigate source system
⚠️ FLAGGED Quantity Nulls: 1 row(s)
   → Reason: Missing quantity values
   → Action: Backfill from order logs or default to 0
⚠️ FLAGGED Name Issues: 1 row(s)
   → Reason: Non-title-case customer name
   → Action: Standardise at ingestion: .title() on all names
⚠️ FLAGGED Category Issues: 1 row(s)
   → Reason: Unlisted or lowercase category
   → Action: Add validation at point of entry


---
### Task D — Pipeline Function (10 pts)


#### D1 — Cleaning Pipeline (10 pts)

Write a function `clean_transactions(transactions)` that:

1. Works on a **copy** of the input (never modifies original)
2. Removes rows where revenue is None or negative
3. Title-cases all customer names
4. Title-cases all category values
5. Fills missing quantity with `0`
6. Returns the cleaned list

Then verify your pipeline by:
- Calling `score_data_quality()` on raw_transactions → show before score
- Calling `clean_transactions()` on raw_transactions → produce cleaned list
- Calling `score_data_quality()` on cleaned list → show after score
- Printing both scores side by side

Expected: quality score should improve from before to after.


In [19]:
# D1 — clean_transactions pipeline
def clean_transactions(transactions):
    """
    Apply a cleaning pipeline to the transaction list.
    
    Changes made (on a copy):
    - Remove rows with None or negative revenue
    - Title-case customer names
    - Title-case category values
    - Fill missing quantity with 0
    
    Parameters
    ----------
    transactions : list of dict — original data (not modified)
    
    Returns
    -------
    list of dict — cleaned copy
    """
    import copy
    # Always work on a deep copy — never touch original
    data = copy.deepcopy(transactions)
    
    cleaned = []
    for row in data:
        # Remove invalid revenue rows
        if row["revenue"] is None or row["revenue"] < 0:
            continue  # skip this row entirely
        
        # Standardise name casing
        row["customer"] = row["customer"].title()
        
        # Standardise category casing
        row["category"] = row["category"].title()
        
        # Fill missing quantity
        if row["quantity"] is None:
            row["quantity"] = 0
        
        cleaned.append(row)
    
    return cleaned


# ── Pipeline verification ──
before = score_data_quality(raw_transactions)
cleaned = clean_transactions(raw_transactions)
after  = score_data_quality(cleaned)

print(f"Rows:          {before['total_rows']} → {after['total_rows']} (removed {before['total_rows'] - after['total_rows']} bad revenue rows)")
print(f"Quality score: {before['quality_score']}% → {after['quality_score']}%")
print(f"Revenue issues:{before['revenue_issues']} → {after['revenue_issues']}")
print(f"Quantity nulls:{before['quantity_nulls']} → {after['quantity_nulls']}")
print(f"Name issues:   {before['name_issues']} → {after['name_issues']}")
print(f"Category issues:{before['category_issues']} → {after['category_issues']}")
print()
print("✅ Original raw_transactions unchanged — verify:")
print(f"   raw_transactions[10]['quantity'] = {raw_transactions[10]['quantity']}  (should still be None)")
print(f"   raw_transactions[11]['customer'] = {raw_transactions[11]['customer']}  (should still be 'meera')")




Rows:          15 → 14 (removed 1 bad revenue rows)
Quality score: 73.3% → 100.0%
Revenue issues:1 → 0
Quantity nulls:1 → 0
Name issues:   1 → 0
Category issues:1 → 0

✅ Original raw_transactions unchanged — verify:
   raw_transactions[10]['quantity'] = None  (should still be None)
   raw_transactions[11]['customer'] = meera  (should still be 'meera')


---
### Task E — Bonus: Recursive Function (5 pts ★)

#### E1 — Factorial and Fibonacci (5 pts)

Write two recursive functions:
1. `factorial(n)` — returns n! (n factorial). Handle n=0 as base case (0! = 1).
2. `fibonacci(n)` — returns the nth Fibonacci number. F(0)=0, F(1)=1.

Test: factorial(0), factorial(5), factorial(10), fibonacci(0), fibonacci(7), fibonacci(10)

No loops allowed — pure recursion only.


In [20]:
# E1 — Recursive functions
def factorial(n):
    """
    Return n! (n factorial) using recursion.
    Base case: factorial(0) = 1
    """
    # Base case
    if n == 0:
        return 1
    # Recursive case
    return n * factorial(n - 1)


def fibonacci(n):
    """
    Return the nth Fibonacci number using recursion.
    F(0) = 0, F(1) = 1, F(n) = F(n-1) + F(n-2)
    """
    # Base cases
    if n == 0:
        return 0
    if n == 1:
        return 1
    # Recursive case
    return fibonacci(n - 1) + fibonacci(n - 2)


# Test
print("Factorial:")
for n in [0, 1, 5, 10]:
    print(f"  factorial({n}) = {factorial(n)}")

print("\nFibonacci:")
for n in [0, 1, 5, 7, 10]:
    print(f"  fibonacci({n}) = {fibonacci(n)}")




Factorial:
  factorial(0) = 1
  factorial(1) = 1
  factorial(5) = 120
  factorial(10) = 3628800

Fibonacci:
  fibonacci(0) = 0
  fibonacci(1) = 1
  fibonacci(5) = 5
  fibonacci(7) = 13
  fibonacci(10) = 55


---
## 📊 Section 5 — Scoring Rubric (80 pts + 5★ bonus)

| Task | Max pts | What earns full marks |
|------|---------|----------------------|
| **A1** classify_revenue | **8** | Handles all 6 test values correctly incl. None + negative; has docstring; returns (not prints) |
| **A2** avg_revenue | **7** | Correct for all-category and filtered; case-insensitive; skips None/negative; returns rounded float |
| **A3** format_row | **10** | Correct padding, comma formatting, currency prefix, ljust to width; returns string |
| **B1** aggregate_metrics | **12** | Accepts *args; correct sum/mean/min/max per field; skips None/negative; returns dict |
| **B2** build_summary_report | **13** | Accepts **kwargs; correct multi-section format; returns string (caller prints); tested with 3+ sections |
| **C1** score_data_quality + print_quality_report | **20** | Finds all 4 intentional issues; correct quality score formula; NRA format in report |
| **D1** clean_transactions pipeline | **10** | Deep copy used; all 4 cleaning steps; before/after quality scores shown; original unchanged (verified) |
| **E1** Bonus — recursion | **5★** | Both functions correct; pure recursion (no loops); base cases handled |

---

### Automatic deductions
- `-2` per function that uses `print` instead of `return` for the main output
- `-2` per missing docstring
- `-3` if original `raw_transactions` is modified anywhere

---

### Interview Question for Day 34

> *"You've written a data cleaning function. A colleague runs it and says the original data changed. What went wrong and how do you fix it?"*

**Model answer:** The function modified the input list in-place instead of working on a copy. Python lists are passed by reference — changes inside the function affect the original unless you explicitly copy. The fix is `import copy; data = copy.deepcopy(input)` at the start of the function, or using `[row.copy() for row in input]` for a shallow copy when rows are flat dicts.

---


---
## 📈 Month 3 Running Scorecard

| Day | Topic | Score | Stars |
|-----|-------|-------|-------|
| Day 32 | Python Basics | 78/80 | — |
| Day 33 | Loops, Conditionals, Functions | 65/65 | — |
| Day 34 | **Functions (today)** | *pending* | *pending* |
| Day 35 | Pandas Introduction | 65/65 | — |
| Day 37 | Pandas Data Cleaning | 79/80 | ⭐ |
| Day 38 | GroupBy & Aggregation | 80/80 | ⭐ |
| Day 39 | Merging & Joining | 85/85 | — |
| Day 40 | Matplotlib & Seaborn | 88/90 | — |
| Day 42 | Advanced Visualization | 98/100 | ⭐ |
| Day 43 | EDA Framework | 100/100 | ⭐ |
| Day 51 | EDA Capstone | 97/100 | ⭐ |

---
**Next after Day 34:** Day 36 — Week 1 Mini-Project (60 pts, Python + Pandas)
